In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os

SUBJECT = "MM15"

# !mkdir -p /content/kara_one
# !cp "/content/drive/MyDrive/kara_one/{SUBJECT}.tar.bz2" /content/kara_one/

In [ ]:
# %cd /content/kara_one
# !tar -xvjf {SUBJECT}.tar.bz2

In [ ]:
!pip install mne==1.6.1

In [ ]:
import glob
import mne

BASE_PATH = f"/content/kara_one/p/spoclab/users/szhao/EEG/data/{SUBJECT}"

cnt_files = glob.glob(BASE_PATH + "/*.cnt")
print(cnt_files)

cnt_path = cnt_files[0]

raw = mne.io.read_raw_cnt(cnt_path, preload=True)
raw.pick_types(eeg=True)

print(raw)

In [ ]:
#raw.plot(n_channels=10, duration=5)

In [ ]:
import scipy.io as sio
import os

epoch_file = os.path.join(BASE_PATH, "epoch_inds.mat")
label_file = os.path.join(BASE_PATH, "kinect_data/labels.txt")

epoch_data = sio.loadmat(epoch_file)

# FIXED
thinking_inds = epoch_data['thinking_inds'][0]

# Load labels
with open(label_file) as f:
    labels = [line.strip() for line in f.readlines()]

print("Total labels:", len(labels))
print("Thinking trials:", len(thinking_inds))

In [ ]:
sfreq = int(raw.info["sfreq"])
epoch_length = sfreq

X = []
clean_labels = []

for i, idx in enumerate(thinking_inds):
    start = int(idx[0][0])   # ✅ FIXED
    stop = start + epoch_length

    if stop <= raw.n_times:
        epoch = raw.get_data(start=start, stop=stop)
        X.append(epoch)
        clean_labels.append(labels[i])

import numpy as np

X = np.array(X)
y = np.array(clean_labels)

print("EEG shape:", X.shape)
print("Labels shape:", y.shape)

In [ ]:
X_cnn = X[:, :, :, np.newaxis]

print("CNN input shape:", X_cnn.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense

cnn_model = Sequential([
    Conv2D(8, (3,3), activation='relu', padding='same', input_shape=(68,1000,1)),
    Flatten(),
    Dense(64, activation='relu')
])

cnn_model.summary()

In [ ]:
cnn_model.predict(X_cnn[:5])

In [ ]:
features = cnn_model.predict(X_cnn)

print("Feature shape:", features.shape)

In [ ]:
import numpy as np

np.save(f"/content/drive/MyDrive/kara_one/{SUBJECT}_features.npy", features)
np.save(f"/content/drive/MyDrive/kara_one/{SUBJECT}_labels.npy", y)

print(f"{SUBJECT} features saved successfully!")

In [ ]:
import numpy as np

subjects = ["MM05", "MM08", "MM09", "MM10", "MM11", "MM15"]

X_list = []
y_list = []

for sub in subjects:
    X_sub = np.load(f"/content/drive/MyDrive/kara_one/{sub}_features.npy")
    y_sub = np.load(f"/content/drive/MyDrive/kara_one/{sub}_labels.npy")

    X_list.append(X_sub)
    y_list.append(y_sub)

X_all = np.concatenate(X_list, axis=0)
y_all = np.concatenate(y_list, axis=0)

print("Combined feature shape:", X_all.shape)
print("Combined labels shape:", y_all.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y_all)

print("Number of classes:", len(le.classes_))

In [ ]:
X_seq = X_all.reshape(X_all.shape[0], 64, 1)

print(X_seq.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_encoded, test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(64,1)),
    Dense(len(le.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=16,
    validation_split=0.2
)

In [ ]:
y_pred = model.predict(X_test)

import numpy as np

y_pred_labels = np.argmax(y_pred, axis=1)

from sklearn.metrics import accuracy_score

print("Top-1 Accuracy:", accuracy_score(y_test, y_pred_labels)*100)

In [ ]:
top3 = np.argsort(y_pred, axis=1)[:, -3:]

correct = sum([y_test[i] in top3[i] for i in range(len(y_test))])

print("Top-3 Accuracy:", correct / len(y_test)*100)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, LayerNormalization, MultiHeadAttention,
    GlobalAveragePooling1D, Dropout
)
from tensorflow.keras.models import Model

input_layer = Input(shape=(64, 1))

# Project input to higher dimension
x = Dense(64)(input_layer)

# Transformer block
attn_output = MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
x = LayerNormalization()(x + attn_output)

# Feed-forward
ff = Dense(128, activation='relu')(x)
ff = Dense(64)(ff)

x = LayerNormalization()(x + ff)

# Pooling
x = GlobalAveragePooling1D()(x)

# Output layer
output = Dense(len(le.classes_), activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=16,
    validation_split=0.2
)

In [ ]:
y_pred = model.predict(X_test)

import numpy as np

y_pred_labels = np.argmax(y_pred, axis=1)

from sklearn.metrics import accuracy_score

print("Top-1 Accuracy:", accuracy_score(y_test, y_pred_labels)*100)

In [ ]:
top3 = np.argsort(y_pred, axis=1)[:, -3:]

correct = sum([y_test[i] in top3[i] for i in range(len(y_test))])

print("Top-3 Accuracy:", correct / len(y_test) * 100)

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
label_embeddings = sbert.encode(le.classes_)

pred_words = le.inverse_transform(y_pred_labels)
true_words = le.inverse_transform(y_test)

pred_embeddings = sbert.encode(pred_words)
true_embeddings = sbert.encode(true_words)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

correct = 0

for i in range(len(y_test)):
    true_word = le.inverse_transform([y_test[i]])[0]

    candidates = le.inverse_transform(top3[i])

    true_emb = sbert.encode([true_word])[0]
    cand_emb = sbert.encode(candidates)

    sims = cosine_similarity([true_emb], cand_emb)[0]

    if max(sims) > 0.5:
        correct += 1

print("Semantic Top-3 Accuracy:", correct / len(y_test) * 100)

In [ ]:
import numpy as np

top3 = np.argsort(y_pred, axis=1)[:, -3:]

top3_words = []

for i in range(len(top3)):
    words = le.inverse_transform(top3[i])
    top3_words.append(words)

# Show first 5 predictions
for i in range(5):
    print(f"Sample {i}:")
    print("Predicted:", top3_words[i])
    print("Actual   :", le.inverse_transform([y_test[i]])[0])
    print()

In [ ]:
import numpy as np
print(np.bincount(y_train))

In [ ]:
import numpy as np

# Get Top-3 indices
top3 = np.argsort(y_pred, axis=1)[:, -3:]

top3_words = []

for i in range(len(top3)):
    words = le.inverse_transform(top3[i])
    top3_words.append(words)

# 🔥 Collect samples with different predictions
selected_indices = []
seen_predictions = set()

for i in range(len(top3_words)):
    pred_tuple = tuple(top3_words[i])  # convert to tuple for set

    if pred_tuple not in seen_predictions:
        seen_predictions.add(pred_tuple)
        selected_indices.append(i)

    if len(selected_indices) >= 5:
        break

# If not enough unique predictions, fill randomly
if len(selected_indices) < 5:
    remaining = list(set(range(len(y_test))) - set(selected_indices))
    extra = np.random.choice(remaining, size=5-len(selected_indices), replace=False)
    selected_indices.extend(extra)

# Show results
for i in selected_indices:
    print(f"Sample {i}:")
    print("Predicted:", top3_words[i])
    print("Actual   :", le.inverse_transform([y_test[i]])[0])
    print()

In [ ]:
def predict_thought(eeg_sample):
    # reshape for CNN
    eeg_sample = eeg_sample[np.newaxis, :, :, np.newaxis]

    # CNN features
    feat = cnn_model.predict(eeg_sample)

    # reshape for transformer
    feat_seq = feat.reshape(1, 64, 1)

    # prediction
    pred = model.predict(feat_seq)

    top3 = np.argsort(pred, axis=1)[:, -3:][0]

    words = le.inverse_transform(top3)

    return words




In [ ]:
sample = X[0]

predicted_words = predict_thought(sample)

print("Predicted words:", predicted_words)
print("Actual word:", y[0])

In [ ]:
top3 = np.argsort(y_pred, axis=1)[:, -3:]

correct = 0

for i in range(len(y_test)):
    true_word = le.inverse_transform([y_test[i]])[0]
    candidates = le.inverse_transform(top3[i])

    true_emb = sbert.encode([true_word])[0]
    cand_emb = sbert.encode(candidates)

    sims = cosine_similarity([true_emb], cand_emb)[0]

    if max(sims) > 0.5:
        correct += 1

final_top3 = correct / len(y_test) * 100

print("FINAL Semantic Top-3 Accuracy:", final_top3)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np

# Predictions
y_pred_labels = np.argmax(y_pred, axis=1)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_labels)

# Plot
plt.figure(figsize=(8,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues', values_format='d')

plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

top1 = accuracy_score(y_test, y_pred_labels) * 100

# Top-3
top3 = np.argsort(y_pred, axis=1)[:, -3:]
top3_acc = np.mean([y_test[i] in top3[i] for i in range(len(y_test))]) * 100

# Semantic Top-3 (your final metric)
semantic_top3 = final_top3

# Plot
labels = ['Top-1', 'Top-3', 'Semantic Top-3']
values = [top1, top3_acc, semantic_top3]

plt.figure()
plt.bar(labels, values)

plt.ylabel("Accuracy (%)")
plt.title("Model Performance Comparison")

for i, v in enumerate(values):
    plt.text(i, v + 1, f"{v:.2f}%", ha='center')

plt.show()

In [ ]:
plt.figure()

# Accuracy
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title("Training vs Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure()

# Accuracy
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title("Training vs Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title("Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = []

for i in range(len(y_test)):
    true_word = le.inverse_transform([y_test[i]])[0]
    pred_word = le.inverse_transform([y_pred_labels[i]])[0]

    true_emb = sbert.encode([true_word])[0]
    pred_emb = sbert.encode([pred_word])[0]

    sim = cosine_similarity([true_emb], [pred_emb])[0][0]
    similarities.append(sim)

plt.figure()
plt.hist(similarities, bins=20)

plt.title("Semantic Similarity Distribution")
plt.xlabel("Similarity Score")
plt.ylabel("Frequency")

plt.show()

In [ ]:
print(le.classes_)

In [ ]:
import numpy as np

# Pick one sample
sample = X_test[0]

# Add batch dimension
sample = np.expand_dims(sample, axis=0)